In [14]:
from google.colab import drive

# Unmount whatever's currently connected, then force a fresh account picker
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

import os

# One fixed root folder for this whole project — persists forever in your Drive
ROOT = "/content/drive/MyDrive/SIH_Kitchen_Hygiene"

INPUT_PHOTOS = os.path.join(ROOT, "input_photos")
INPUT_VIDEOS = os.path.join(ROOT, "input_videos")
OUTPUT_ANNOTATED = os.path.join(ROOT, "output_annotated")

for folder in [INPUT_PHOTOS, INPUT_VIDEOS, OUTPUT_ANNOTATED]:
    os.makedirs(folder, exist_ok=True)

print("Your folders are ready at:")
print(f"  Photos in:    {INPUT_PHOTOS}")
print(f"  Videos in:    {INPUT_VIDEOS}")
print(f"  Outputs in:   {OUTPUT_ANNOTATED}")

Mounted at /content/drive
Your folders are ready at:
  Photos in:    /content/drive/MyDrive/SIH_Kitchen_Hygiene/input_photos
  Videos in:    /content/drive/MyDrive/SIH_Kitchen_Hygiene/input_videos
  Outputs in:   /content/drive/MyDrive/SIH_Kitchen_Hygiene/output_annotated


In [15]:
import glob

def get_input_photos():
    """Returns all source photos, guaranteed to exclude any annotated outputs."""
    extensions = ("*.jpg", "*.jpeg", "*.png")
    paths = []
    for ext in extensions:
        paths.extend(glob.glob(os.path.join(INPUT_PHOTOS, ext)))
        paths.extend(glob.glob(os.path.join(INPUT_PHOTOS, ext.upper())))
    return sorted(paths)

def get_input_videos():
    """Returns all source videos."""
    extensions = ("*.mp4", "*.mov", "*.avi")
    paths = []
    for ext in extensions:
        paths.extend(glob.glob(os.path.join(INPUT_VIDEOS, ext)))
        paths.extend(glob.glob(os.path.join(INPUT_VIDEOS, ext.upper())))
    return sorted(paths)

print(f"Found {len(get_input_photos())} photos, {len(get_input_videos())} videos.")

Found 21 photos, 9 videos.


In [16]:
!pip install -q google-genai pillow

In [17]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("apikey")


In [18]:
from google import genai
from google.genai import types

In [19]:
client = genai.Client()  # create ONCE, keep alive for the whole notebook session

In [20]:
import os
import json
import time
from google import genai
from google.genai import types
from PIL import Image, ImageDraw, ImageFont

# NOTE: run this AFTER your API key + client cell, e.g.:
#   os.environ["GOOGLE_API_KEY"] = userdata.get("apikey")
#   client = genai.Client()
# Everything below reuses that one global `client` — never creates a new one.

# ---------------------------------------------------------------------------
# 0. Model fallback chain — tries each model in order, skips exhausted ones
# ---------------------------------------------------------------------------
MODEL_FALLBACK_CHAIN = [
    "gemini-3.8-flash",
    "gemini-3.1-flash-lite",
    "gemini-3-flash-preview",
    "gemini-2.5-flash",
]

def generate_with_fallback(contents, config, models=MODEL_FALLBACK_CHAIN, max_retries_per_model=2):
    """Try each model in order. Skip immediately to the next model on quota
    exhaustion (429). Retry a few times on temporary overload (503) before
    giving up on that model and moving on. Raises only if EVERY model fails."""
    last_error = None

    for model_name in models:
        for attempt in range(max_retries_per_model):
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=contents,
                    config=config,
                )
                print(f"Succeeded using model: {model_name}")
                return response
            except Exception as e:
                last_error = e
                error_str = str(e)

                if "RESOURCE_EXHAUSTED" in error_str or "429" in error_str:
                    print(f"{model_name}: quota exhausted, moving to next model...")
                    break

                elif "UNAVAILABLE" in error_str or "503" in error_str:
                    if attempt < max_retries_per_model - 1:
                        wait_time = 5 * (attempt + 1)
                        print(f"{model_name}: overloaded, retrying in {wait_time}s...")
                        time.sleep(wait_time)
                    else:
                        print(f"{model_name}: still overloaded after retries, moving to next model...")

                else:
                    print(f"{model_name}: unexpected error ({e}), moving to next model...")
                    break

    raise RuntimeError(f"All models in fallback chain failed. Last error: {last_error}")


def chat_send_with_fallback(chat_factory, send_fn, models=MODEL_FALLBACK_CHAIN, max_retries_per_model=2):
    """Same fallback logic, but for chat sessions. chat_factory(model_name) creates
    a fresh chat on that model; send_fn(chat) sends the message and returns the
    response. Returns (chat, response) from whichever model succeeded first."""
    last_error = None

    for model_name in models:
        for attempt in range(max_retries_per_model):
            try:
                chat = chat_factory(model_name)
                response = send_fn(chat)
                print(f"Succeeded using model: {model_name}")
                return chat, response
            except Exception as e:
                last_error = e
                error_str = str(e)

                if "RESOURCE_EXHAUSTED" in error_str or "429" in error_str:
                    print(f"{model_name}: quota exhausted, moving to next model...")
                    break

                elif "UNAVAILABLE" in error_str or "503" in error_str:
                    if attempt < max_retries_per_model - 1:
                        wait_time = 5 * (attempt + 1)
                        print(f"{model_name}: overloaded, retrying in {wait_time}s...")
                        time.sleep(wait_time)
                    else:
                        print(f"{model_name}: still overloaded after retries, moving to next model...")

                else:
                    print(f"{model_name}: unexpected error ({e}), moving to next model...")
                    break

    raise RuntimeError(f"All models in fallback chain failed. Last error: {last_error}")


# ---------------------------------------------------------------------------
# 1. Response schema — includes bounding_box for every flagged issue
# ---------------------------------------------------------------------------
RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "is_hygienic": {"type": "boolean"},
        "hygiene_score": {"type": "integer", "description": "0-100, 100=spotless"},
        "confidence": {"type": "number"},
        "flagged_issues": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "issue": {"type": "string", "description": "Short label e.g. 'spill_on_floor', 'no_gloves'"},
                    "severity": {"type": "string", "enum": ["low", "medium", "high", "critical"]},
                    "description": {"type": "string"},
                    "bounding_box": {
                        "type": "array",
                        "items": {"type": "integer"},
                        "description": "[ymin, xmin, ymax, xmax], normalized 0-1000 scale, marking exactly where this issue is visible."
                    }
                },
                "required": ["issue", "severity", "description", "bounding_box"]
            }
        },
        "positive_observations": {"type": "array", "items": {"type": "string"}},
        "explanation": {"type": "string"}
    },
    "required": ["is_hygienic", "hygiene_score", "confidence", "flagged_issues", "positive_observations", "explanation"]
}

SYSTEM_PROMPT = """You are a food safety inspector AI analyzing a single photo from a kitchen
surveillance snapshot. Evaluate ONLY what is visible — do not assume anything outside the frame.

Assess against these criteria (modeled on FSSAI/food-safety hygiene audit standards):
1. Surface cleanliness — grime, food debris, stains, spills, residue
2. Clutter — is the workstation organized or are items piled/out of place
3. PPE — hairnets, gloves, aprons on any visible staff
4. Cross-contamination risk — raw food near ready-to-eat food, improper storage
5. Pest signs — visible pests, droppings, infestation indicators
6. Waste management — overflowing bins, improperly stored waste
7. Fire/safety hazards — unattended flames, blocked exits, exposed blades

For EVERY flagged issue, you MUST provide a bounding_box that tightly marks the exact
region of the image where that issue is visible, as [ymin, xmin, ymax, xmax] on a
0-1000 normalized scale (top-left is [0,0], bottom-right is [1000,1000]). Be precise —
the box should tightly frame just the problem area, not the whole image.

Be specific and evidence-based. If ambiguous or blurry, reflect that with lower confidence
rather than guessing. Do not hallucinate issues not visible in the image. Respond ONLY in
the structured JSON format provided."""


def analyze_kitchen_photo(image_path: str) -> dict:
    """Send one photo to Gemini and return a parsed structured verdict with bounding
    boxes, automatically falling back across models if one is unavailable."""
    with open(image_path, "rb") as f:
        image_bytes = f.read()

    ext = image_path.lower().split(".")[-1]
    mime_type = "image/png" if ext == "png" else "image/jpeg"

    response = generate_with_fallback(
        contents=[
            types.Part.from_bytes(data=image_bytes, mime_type=mime_type),
            SYSTEM_PROMPT,
        ],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=RESPONSE_SCHEMA,
            temperature=0.1,
        ),
    )
    return json.loads(response.text)


# ---------------------------------------------------------------------------
# 2. Draw the boxes on the image, YOLO-style
# ---------------------------------------------------------------------------
SEVERITY_COLORS = {
    "low": "#F1C40F",
    "medium": "#E67E22",
    "high": "#E74C3C",
    "critical": "#8E0000",
}

def draw_annotations(image_path: str, result: dict, output_path: str = None) -> str:
    """Draw bounding boxes + labels on the image and save an annotated copy."""
    img = Image.open(image_path).convert("RGB")
    width, height = img.size
    draw = ImageDraw.Draw(img)

    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", size=max(14, width // 60))
    except Exception:
        font = ImageFont.load_default()

    boxes_drawn = 0
    for item in result.get("flagged_issues", []):
        box = item.get("bounding_box")
        if not box or len(box) != 4:
            continue

        ymin, xmin, ymax, xmax = box
        left = (xmin / 1000) * width
        top = (ymin / 1000) * height
        right = (xmax / 1000) * width
        bottom = (ymax / 1000) * height

        color = SEVERITY_COLORS.get(item.get("severity", "low"), "#F1C40F")
        draw.rectangle([left, top, right, bottom], outline=color, width=3)

        label = f"{item['issue']} ({item['severity']})"
        text_bbox = draw.textbbox((left, top), label, font=font)
        draw.rectangle(
            [left, top - (text_bbox[3] - text_bbox[1]) - 6, text_bbox[2] + 6, top],
            fill=color,
        )
        draw.text((left + 3, top - (text_bbox[3] - text_bbox[1]) - 4), label, fill="white", font=font)
        boxes_drawn += 1

    print(f"Boxes drawn: {boxes_drawn} (out of {len(result.get('flagged_issues', []))} flagged issues)")

    if output_path is None:
        base, ext = os.path.splitext(image_path)
        output_path = f"{base}_annotated{ext}"

    img.save(output_path)
    return output_path


# ---------------------------------------------------------------------------
# 3. Q&A follow-up layer
# ---------------------------------------------------------------------------
QA_SYSTEM_PROMPT = """You are the same food safety inspector AI that just analyzed this kitchen
photo. You already produced a structured hygiene report (given below as JSON). A hotel manager
or a judge evaluating this system is now asking you follow-up questions about that specific
analysis.

Rules:
- Answer ONLY based on the photo and the analysis you already produced — don't invent new
  issues that weren't in the original flagged_issues list.
- Be specific and reference the actual severity levels, scores, and issues from the report.
- Keep answers conversational and short — 2-4 sentences.
- If asked something the photo genuinely can't answer, say so honestly rather than guessing.

Here is your original structured analysis of this photo:
{result_json}
"""

def start_qa_session(image_path: str, result: dict):
    """Start a chat session grounded in the photo + its analysis result,
    falling back across models if the preferred one is unavailable."""
    with open(image_path, "rb") as f:
        image_bytes = f.read()

    ext = image_path.lower().split(".")[-1]
    mime_type = "image/png" if ext == "png" else "image/jpeg"

    def chat_factory(model_name):
        return client.chats.create(
            model=model_name,
            config=types.GenerateContentConfig(
                system_instruction=QA_SYSTEM_PROMPT.format(result_json=json.dumps(result, indent=2)),
                temperature=0.3,
            ),
        )

    def send_fn(chat):
        return chat.send_message([
            types.Part.from_bytes(data=image_bytes, mime_type=mime_type),
            "Here is the photo you analyzed. I may ask follow-up questions about it.",
        ])

    chat, _ = chat_send_with_fallback(chat_factory, send_fn)
    chat.image_path = image_path
    chat.result = result
    return chat


def ask(chat, question: str) -> str:
    """Ask a follow-up question in an existing Q&A session, with fallback
    to a fresh session on a different model if the current one fails."""
    try:
        response = chat.send_message(question)
        return response.text
    except Exception as e:
        print(f"Current session's model failed ({e}). Starting a fresh session on a fallback model...")
        new_chat = start_qa_session(chat.image_path, chat.result)
        response = new_chat.send_message(question)
        # copy the new chat's state back so future ask() calls keep working on it
        chat.__dict__.update(new_chat.__dict__)
        return response.text


def verify_session(chat):
    """Sanity check: shows exactly which photo, boxes, and result this session is grounded in."""
    print("This session is grounded in:", chat.image_path)
    print("\nFlagged issues + bounding boxes:")
    for item in chat.result.get("flagged_issues", []):
        print(f"  {item['issue']} ({item['severity']}): {item.get('bounding_box')}")
    annotated_path = draw_annotations(chat.image_path, chat.result)
    return Image.open(annotated_path)

In [21]:
import os
import glob

def run_batch_analysis(folder_path: str, output_folder: str = None):
    if output_folder is None:
        output_folder = os.path.join(folder_path, "annotated")
    os.makedirs(output_folder, exist_ok=True)

    image_extensions = ("*.jpg", "*.jpeg", "*.png")
    image_paths = []
    for ext in image_extensions:
        image_paths.extend(glob.glob(os.path.join(folder_path, ext)))
        image_paths.extend(glob.glob(os.path.join(folder_path, ext.upper())))

    print(f"Found {len(image_paths)} images to process.\n")

    batch_results = []
    for i, image_path in enumerate(sorted(image_paths), 1):
        filename = os.path.basename(image_path)
        print(f"[{i}/{len(image_paths)}] Processing {filename}...")
        try:
            result = analyze_kitchen_photo(image_path)
            annotated_path = draw_annotations(
                image_path, result,
                output_path=os.path.join(output_folder, f"annotated_{filename}")
            )
            batch_results.append({
                "filename": filename, "image_path": image_path,
                "annotated_path": annotated_path,
                "hygiene_score": result["hygiene_score"],
                "is_hygienic": result["is_hygienic"],
                "confidence": result["confidence"],
                "num_issues": len(result["flagged_issues"]),
                "issues": [f["issue"] for f in result["flagged_issues"]],
                "result": result,
            })
            print(f"    Score: {result['hygiene_score']}/100, Issues: {len(result['flagged_issues'])}\n")
        except Exception as e:
            print(f"    FAILED: {e}\n")
            batch_results.append({"filename": filename, "image_path": image_path, "error": str(e)})

    return batch_results

In [22]:
results = run_batch_analysis(INPUT_PHOTOS, output_folder=OUTPUT_ANNOTATED)

import pandas as pd
summary = pd.DataFrame([
    {
        "Photo": r["filename"],
        "Score": r.get("hygiene_score", "ERROR"),
        "Hygienic?": r.get("is_hygienic", "-"),
        "Confidence": r.get("confidence", "-"),
        "# Issues": r.get("num_issues", "-"),
    }
    for r in results
])
summary

Found 21 images to process.

[1/21] Processing Screenshot 2026-09-27 235237.png...
gemini-3.8-flash: overloaded, retrying in 5s...
gemini-3.8-flash: still overloaded after retries, moving to next model...
Succeeded using model: gemini-3.1-flash-lite
Boxes drawn: 3 (out of 3 flagged issues)
    Score: 35/100, Issues: 3

[2/21] Processing Screenshot 2026-09-27 235251.png...
Succeeded using model: gemini-3.8-flash
Boxes drawn: 6 (out of 6 flagged issues)
    Score: 25/100, Issues: 6

[3/21] Processing Screenshot 2026-09-27 235307.png...
gemini-3.8-flash: overloaded, retrying in 5s...
gemini-3.8-flash: still overloaded after retries, moving to next model...
Succeeded using model: gemini-3.1-flash-lite
Boxes drawn: 2 (out of 2 flagged issues)
    Score: 55/100, Issues: 2

[4/21] Processing Screenshot 2026-09-27 235319.png...
gemini-3.8-flash: overloaded, retrying in 5s...
gemini-3.8-flash: quota exhausted, moving to next model...
gemini-3.1-flash-lite: overloaded, retrying in 5s...
Succeede

,Photo,Score,Hygienic?,Confidence,# Issues
0,Screenshot 2026-09-27 235237.png,35,False,0.95,3
1,Screenshot 2026-09-27 235251.png,25,False,0.95,6
2,Screenshot 2026-09-27 235307.png,55,False,0.95,2
3,Screenshot 2026-09-27 235319.png,0,False,1.0,3
4,Screenshot 2026-09-27 235329.png,35,False,0.95,3
5,Screenshot 2026-09-27 235341.png,25,False,0.95,3
6,Screenshot 2026-09-27 235355.png,5,False,0.98,5
7,Screenshot 2026-09-27 235409.png,15,False,0.95,4
8,Screenshot 2026-09-27 235422.png,5,False,0.98,2
9,Screenshot 2026-09-27 235446.png,35,False,0.95,4


vedio analysis


In [23]:
"""
Video Analysis Layer — extracts frames from kitchen videos and runs the
SAME analyze_kitchen_photo() pipeline on them, then aggregates the results
into one verdict per video (worst-case scoring, since a video with even one
bad moment should not be scored as if that moment never happened).

Requires: opencv-python (usually pre-installed in Colab as `cv2`)
"""

import os
import cv2
import numpy as np


def extract_frames(video_path: str, output_dir: str, num_frames: int = 3) -> list:
    """Extracts `num_frames` evenly-spaced frames from a video and saves them
    as .jpg files. Returns the list of saved frame paths."""
    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        raise RuntimeError(f"Video has no readable frames: {video_path}")

    # Evenly space frame indices across the video, skipping the very start/end
    # (often black frames or transitions)
    indices = np.linspace(total_frames * 0.05, total_frames * 0.95, num_frames, dtype=int)

    base_name = os.path.splitext(os.path.basename(video_path))[0]
    saved_paths = []

    for i, frame_idx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        success, frame = cap.read()
        if not success:
            continue

        frame_path = os.path.join(output_dir, f"{base_name}_frame{i+1}.jpg")
        cv2.imwrite(frame_path, frame)
        saved_paths.append(frame_path)

    cap.release()

    if not saved_paths:
        raise RuntimeError(f"Failed to extract any frames from: {video_path}")

    return saved_paths


def sharpest_frame(frame_paths: list) -> str:
    """Returns the path of the least-blurry frame, using Laplacian variance
    (a standard, fast blur-detection technique). Higher variance = sharper."""
    best_path, best_score = frame_paths[0], -1
    for path in frame_paths:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        variance = cv2.Laplacian(img, cv2.CV_64F).var()
        if variance > best_score:
            best_score = variance
            best_path = path
    return best_path


def analyze_kitchen_video(video_path: str, output_dir: str, num_frames: int = 3) -> dict:
    """Extracts frames from a video, analyzes EACH with analyze_kitchen_photo(),
    and aggregates them into one worst-case verdict for the whole video.

    Aggregation logic (safety-first, not average-first):
    - hygiene_score: the LOWEST score across all frames (one bad moment counts)
    - flagged_issues: union of all issues seen across frames, deduplicated,
      keeping the highest severity seen for each issue
    - positive_observations: only kept if EVERY frame reported them (i.e.
      consistently clean, not just clean in one lucky frame)
    - confidence: average across frames
    - representative image: the sharpest frame, annotated with the union
      of issues, used as the video's thumbnail on the map
    """
    frame_paths = extract_frames(video_path, output_dir, num_frames=num_frames)

    frame_results = []
    for frame_path in frame_paths:
        try:
            result = analyze_kitchen_photo(frame_path)
            frame_results.append((frame_path, result))
        except Exception as e:
            print(f"  Frame {os.path.basename(frame_path)} failed: {e}")

    if not frame_results:
        raise RuntimeError(f"All frames failed analysis for: {video_path}")

    # --- Worst-case hygiene score ---
    worst_score = min(r["hygiene_score"] for _, r in frame_results)
    is_hygienic = worst_score >= 75

    # --- Union of flagged issues, deduplicated by keeping worst severity ---
    severity_rank = {"low": 1, "medium": 2, "high": 3, "critical": 4}
    issues_by_name = {}
    for _, r in frame_results:
        for issue in r["flagged_issues"]:
            existing = issues_by_name.get(issue["issue"])
            if not existing or severity_rank.get(issue["severity"], 0) > severity_rank.get(existing["severity"], 0):
                issues_by_name[issue["issue"]] = issue
    merged_issues = list(issues_by_name.values())

    # --- Positive observations only if consistent across every frame ---
    all_obs_sets = [set(r["positive_observations"]) for _, r in frame_results]
    consistent_obs = list(set.intersection(*all_obs_sets)) if all_obs_sets else []

    # --- Average confidence ---
    avg_confidence = sum(r["confidence"] for _, r in frame_results) / len(frame_results)

    # --- Representative frame: sharpest one, annotated with merged issues ---
    rep_frame_path = sharpest_frame([fp for fp, _ in frame_results])
    rep_result_for_drawing = {"flagged_issues": merged_issues}
    annotated_path = draw_annotations(
        rep_frame_path, rep_result_for_drawing,
        output_path=os.path.join(output_dir, f"{os.path.splitext(os.path.basename(video_path))[0]}_annotated.jpg")
    )

    aggregated_result = {
        "hygiene_score": worst_score,
        "is_hygienic": is_hygienic,
        "confidence": round(avg_confidence, 2),
        "flagged_issues": merged_issues,
        "positive_observations": consistent_obs,
        "explanation": f"Aggregated from {len(frame_results)} frames sampled across the video. "
                        f"Worst-case score used: {worst_score}/100.",
    }

    return {
        "filename": os.path.basename(video_path),
        "image_path": rep_frame_path,
        "annotated_path": annotated_path,
        "hygiene_score": worst_score,
        "is_hygienic": is_hygienic,
        "confidence": aggregated_result["confidence"],
        "num_issues": len(merged_issues),
        "issues": [i["issue"] for i in merged_issues],
        "result": aggregated_result,
    }


def run_batch_video_analysis(video_folder: str, output_folder: str, num_frames: int = 3) -> list:
    """Runs analyze_kitchen_video() on every video in a folder. Returns
    results in the SAME shape as run_batch_analysis(), so you can feed the
    output directly into export_kitchens_json() alongside your photo results."""
    os.makedirs(output_folder, exist_ok=True)
    video_paths = get_input_videos()  # uses your existing helper

    print(f"Found {len(video_paths)} videos to process.\n")

    batch_results = []
    for i, video_path in enumerate(video_paths, 1):
        filename = os.path.basename(video_path)
        print(f"[{i}/{len(video_paths)}] Processing {filename}...")
        try:
            result = analyze_kitchen_video(video_path, output_folder, num_frames=num_frames)
            print(f"    Worst-case score: {result['hygiene_score']}/100, Issues: {result['num_issues']}\n")
            batch_results.append(result)
        except Exception as e:
            print(f"    FAILED: {e}\n")
            batch_results.append({"filename": filename, "error": str(e)})

    return batch_results

In [24]:
video_results = run_batch_video_analysis(INPUT_VIDEOS, OUTPUT_ANNOTATED, num_frames=3)

# Combine with your photo results and export everything to one map:
photo_results = run_batch_analysis(INPUT_PHOTOS, output_folder=OUTPUT_ANNOTATED)
all_results = photo_results + video_results
kitchens_data = export_kitchens_json(all_results)

Found 9 videos to process.

[1/9] Processing 12328659-hd_1920_1080_24fps.mp4...
gemini-3.8-flash: quota exhausted, moving to next model...
Succeeded using model: gemini-3.1-flash-lite
gemini-3.8-flash: quota exhausted, moving to next model...
Succeeded using model: gemini-3.1-flash-lite
gemini-3.8-flash: quota exhausted, moving to next model...
Succeeded using model: gemini-3.1-flash-lite
Boxes drawn: 5 (out of 5 flagged issues)
    Worst-case score: 35/100, Issues: 5

[2/9] Processing 12333567_2160_3840_29fps.mp4...
gemini-3.8-flash: quota exhausted, moving to next model...
gemini-3.1-flash-lite: overloaded, retrying in 5s...
gemini-3.1-flash-lite: still overloaded after retries, moving to next model...
gemini-3-flash-preview: overloaded, retrying in 5s...
gemini-3-flash-preview: still overloaded after retries, moving to next model...
gemini-2.5-flash: unexpected error (404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new 

In [ ]:
import matplotlib.pyplot as plt

successful = [r for r in results if "annotated_path" in r]
cols = 3
rows = (len(successful) + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes

for ax, r in zip(axes, successful):
    img = Image.open(r["annotated_path"])
    ax.imshow(img)
    ax.set_title(f"{r['filename']}\nScore: {r['hygiene_score']}", fontsize=10)
    ax.axis("off")

for ax in axes[len(successful):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
DEFAULT_DEMO_QUESTIONS = [
    "Why did this get the severity level it did?",
    "What would happen if this wasn't fixed before service?",
    "Is this photo good enough to make this judgment confidently?",
]

def run_batch_analysis_with_qa(folder_path: str, questions: list = None, output_folder: str = None):
    """Like run_batch_analysis, but also runs a fixed set of Q&A questions
    against EVERY photo, storing the answers alongside the analysis.

    WARNING: this uses many more API requests than run_batch_analysis alone —
    1 (analysis) + len(questions) requests PER photo. 5 photos x 3 questions
    = 20 requests just for this step. Watch your quota."""
    if questions is None:
        questions = DEFAULT_DEMO_QUESTIONS

    if output_folder is None:
        output_folder = os.path.join(folder_path, "annotated")
    os.makedirs(output_folder, exist_ok=True)

    image_extensions = ("*.jpg", "*.jpeg", "*.png")
    image_paths = []
    for ext in image_extensions:
        image_paths.extend(glob.glob(os.path.join(folder_path, ext)))
        image_paths.extend(glob.glob(os.path.join(folder_path, ext.upper())))

    print(f"Found {len(image_paths)} images. Will make ~{len(image_paths) * (1 + len(questions))} API calls total.\n")

    batch_results = []
    for i, image_path in enumerate(sorted(image_paths), 1):
        filename = os.path.basename(image_path)
        print(f"[{i}/{len(image_paths)}] Analyzing {filename}...")

        try:
            result = analyze_kitchen_photo(image_path)
            annotated_path = draw_annotations(
                image_path, result,
                output_path=os.path.join(output_folder, f"annotated_{filename}")
            )
            print(f"    Score: {result['hygiene_score']}/100, Issues: {len(result['flagged_issues'])}")

            print(f"    Starting Q&A session...")
            session = start_qa_session(image_path, result)

            qa_answers = {}
            for q in questions:
                print(f"      Q: {q}")
                answer = ask(session, q)
                qa_answers[q] = answer
                print(f"      A: {answer[:80]}{'...' if len(answer) > 80 else ''}")

            batch_results.append({
                "filename": filename,
                "image_path": image_path,
                "annotated_path": annotated_path,
                "hygiene_score": result["hygiene_score"],
                "is_hygienic": result["is_hygienic"],
                "confidence": result["confidence"],
                "num_issues": len(result["flagged_issues"]),
                "issues": [f["issue"] for f in result["flagged_issues"]],
                "result": result,
                "qa_answers": qa_answers,
            })
            print()

        except Exception as e:
            print(f"    FAILED: {e}\n")
            batch_results.append({"filename": filename, "image_path": image_path, "error": str(e)})

    return batch_results

In [ ]:
full_results = run_batch_analysis_with_qa(INPUT_PHOTOS, output_folder=OUTPUT_ANNOTATED)

In [ ]:
for r in full_results:
    if "error" in r:
        print(f"❌ {r['filename']}: {r['error']}\n")
        continue

    print(f"{'='*70}")
    print(f"📷 {r['filename']}  |  Score: {r['hygiene_score']}/100  |  Confidence: {r['confidence']}")
    print(f"{'='*70}")
    print("Flagged issues:", ", ".join(r["issues"]) if r["issues"] else "None")
    print(f"\nExplanation: {r['result']['explanation']}\n")

    print("Q&A:")
    for q, a in r["qa_answers"].items():
        print(f"  Q: {q}")
        print(f"  A: {a}\n")

    display(Image.open(r["annotated_path"]))
    print("\n")

In [ ]:
import os
import json
import shutil

# Sample locations auto-assigned in order to each analyzed photo. Includes
# address + cuisine now, used by the control-center detail panel.
SAMPLE_LOCATIONS = [
    {"name": "Hotel Grand Meridian", "lat": 22.7196, "lng": 75.8577, "address": "12 MG Road, Indore", "cuisine": "North Indian"},
    {"name": "Spice Route Restaurant", "lat": 22.7280, "lng": 75.8650, "address": "45 Race Course Road, Indore", "cuisine": "Multi-cuisine"},
    {"name": "Riverside Kitchen & Bar", "lat": 22.7100, "lng": 75.8450, "address": "7 Riverside Colony, Indore", "cuisine": "Continental"},
    {"name": "The Copper Ladle", "lat": 22.7340, "lng": 75.8390, "address": "22 Airport Road, Indore", "cuisine": "Cafe"},
    {"name": "Old Town Diner", "lat": 22.7050, "lng": 75.8700, "address": "3 Old Town Square, Indore", "cuisine": "Fast food"},
    {"name": "Green Leaf Cafe", "lat": 22.7220, "lng": 75.8300, "address": "18 Sapna Sangeeta Road, Indore", "cuisine": "Vegetarian"},
]

# Optional: force a SPECIFIC photo filename to a SPECIFIC hotel. Leave {} to auto-assign.
KITCHEN_LOCATIONS = {}


def export_kitchens_json(batch_results: list, output_path: str = None, copy_images: bool = True) -> list:
    """Converts batch analysis results into the richer JSON shape the
    control-center map expects, including a reference to each photo's
    annotated image (copied alongside kitchens.json for easy portability)."""
    if output_path is None:
        output_path = os.path.join(OUTPUT_ANNOTATED, "kitchens.json")
    output_dir = os.path.dirname(output_path)

    kitchens = []
    pool_index = 0

    for r in batch_results:
        if "error" in r:
            continue

        location = KITCHEN_LOCATIONS.get(r["filename"])
        if not location:
            location = SAMPLE_LOCATIONS[pool_index % len(SAMPLE_LOCATIONS)]
            pool_index += 1

        # Copy the annotated image next to kitchens.json (if not already there)
        # so the map can reference it with a simple relative filename.
        image_filename = None
        if "annotated_path" in r:
            image_filename = os.path.basename(r["annotated_path"])
            dest = os.path.join(output_dir, image_filename)
            if copy_images and os.path.abspath(r["annotated_path"]) != os.path.abspath(dest):
                shutil.copy(r["annotated_path"], dest)

        result = r["result"]
        kitchens.append({
            "name": location["name"],
            "lat": location["lat"],
            "lng": location["lng"],
            "address": location["address"],
            "cuisine": location["cuisine"],
            "score": result["hygiene_score"],
            "confidence": result["confidence"],
            "explanation": result["explanation"],
            "flagged_issues": result["flagged_issues"],       # full objects: issue, severity, description
            "positive_observations": result["positive_observations"],
            "image": image_filename,
            "lastChecked": "Just now",
        })

    with open(output_path, "w") as f:
        json.dump(kitchens, f, indent=2)

    print(f"Saved {len(kitchens)} kitchens (with images) to {output_path}")
    return kitchens

In [ ]:
results = run_batch_analysis(INPUT_PHOTOS, output_folder=OUTPUT_ANNOTATED)
kitchens_data = export_kitchens_json(results)

In [ ]:
kitchens_data = export_kitchens_json(results)

In [33]:
print(get_input_photos())
6print(get_input_videos())

SyntaxError: invalid decimal literal (3692886517.py, line 2)